# AusLAMP Victoria: QA/QC and Site Analysis

**Goal**: Comprehensive quality control and visualization of all 100 MT sites.

**Analysis includes**:
1. Load metadata for all sites
2. Site location maps (EDL vs LEMI-424)
3. Recording timelines for remote reference selection
4. Statistical analysis (first hour of each site)
5. AGRF25 model comparison
6. Data quality flags and recommendations

**Data**: 81 EDL sites (2014) + 19 LEMI-424 sites (2016-2017)

In [1]:
import sys
sys.path.insert(0, '../src')

from readers import read_station, get_all_metadata
from config import DataPaths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

def caption(text):
    """Display a figure caption as wrapped, italic markdown."""
    from IPython.display import Markdown, display
    display(Markdown(f"*{text}*"))

## 1. Load Metadata for All Sites

In [ ]:
# Load from CSV (already generated)
metadata_file = DataPaths.PROCESSED_DATA_DIR / 'metadata_all_sites.csv'
df_meta = pd.read_csv(metadata_file)

# Parse datetime columns (format='mixed' handles inconsistent timezone formats)
df_meta['start_time'] = pd.to_datetime(df_meta['start_time'], format='mixed')
df_meta['end_time'] = pd.to_datetime(df_meta['end_time'], format='mixed')

print(f"Loaded metadata for {len(df_meta)} sites")
print(f"  EDL: {len(df_meta[df_meta['instrument']=='EDL'])}")
print(f"  LEMI-424: {len(df_meta[df_meta['instrument']=='LEMI-424'])}")

df_meta.head(10)

In [ ]:
# Summary statistics
print("\nRecording Duration by Instrument:")
print(df_meta.groupby('instrument')['duration_days'].describe())

print("\nDate Range:")
print(f"  Earliest: {df_meta['start_time'].min()}")
print(f"  Latest: {df_meta['end_time'].max()}")

print("\nSample Rates:")
print(df_meta.groupby('instrument')['sample_rate'].value_counts())

## 2. Site Location Map

In [ ]:
# Create map with Cartopy
fig = plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree())

# Set extent to Victoria
lat_min = df_meta['latitude'].min() - 0.5
lat_max = df_meta['latitude'].max() + 0.5
lon_min = df_meta['longitude'].min() - 0.5
lon_max = df_meta['longitude'].max() + 0.5
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Add features
ax.add_feature(cfeature.LAND, facecolor='wheat', alpha=0.3)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, edgecolor='black', linewidth=1.0)
ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.RIVERS, linewidth=0.5)

# Plot EDL sites
edl_sites = df_meta[df_meta['instrument'] == 'EDL']
ax.scatter(edl_sites['longitude'], edl_sites['latitude'],
           c='blue', marker='o', s=50, alpha=0.7, 
           label=f'EDL (n={len(edl_sites)})',
           transform=ccrs.PlateCarree(), zorder=5)

# Plot LEMI-424 sites
lemi_sites = df_meta[df_meta['instrument'] == 'LEMI-424']
ax.scatter(lemi_sites['longitude'], lemi_sites['latitude'],
           c='red', marker='^', s=80, alpha=0.7,
           label=f'LEMI-424 (n={len(lemi_sites)})',
           transform=ccrs.PlateCarree(), zorder=5)

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.legend(loc='upper right', fontsize=11)
ax.set_title('AusLAMP Victoria: MT Site Locations', fontsize=14, fontweight='bold')

plt.tight_layout()

# Save figure
fig.savefig(DataPaths.FIGURES_DIR / 'site_locations_map.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 1. Map showing {len(df_meta)} MT sites across Victoria. "
        f"Blue circles: EDL sites (n={len(edl_sites)}, 2014). "
        f"Red triangles: LEMI-424 sites (n={len(lemi_sites)}, 2016-2017).")

## 3. Recording Timelines for Remote Reference Selection

In [ ]:
# EDL Recording Timeline
edl_data = df_meta[df_meta['instrument'] == 'EDL'].copy()
edl_data = edl_data.sort_values('start_time')

fig, ax = plt.subplots(figsize=(16, 12))

for idx, (_, row) in enumerate(edl_data.iterrows()):
    ax.barh(idx, 
            (row['end_time'] - row['start_time']).total_seconds() / 86400,
            left=row['start_time'], 
            height=0.8, 
            color='blue', 
            alpha=0.6,
            edgecolor='darkblue',
            linewidth=0.5)
    
    # Add site name label
    ax.text(row['start_time'], idx, f"  {row['station_id']}", 
            va='center', ha='right', fontsize=7, color='black')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Site', fontsize=12, fontweight='bold')
ax.set_title('EDL Recording Timeline (for Remote Reference Selection)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')
ax.set_yticks(range(len(edl_data)))
ax.set_yticklabels(edl_data['station_id'].values, fontsize=7)

# Format x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
fig.savefig(DataPaths.FIGURES_DIR / 'edl_recording_timeline.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 2. EDL recording timeline showing {len(edl_data)} sites sorted by start date. "
        f"Horizontal bars represent recording periods. Overlapping recordings can be used "
        f"as remote references for magnetotelluric processing.")

In [ ]:
# LEMI-424 Recording Timeline
lemi_data = df_meta[df_meta['instrument'] == 'LEMI-424'].copy()
lemi_data = lemi_data.sort_values('start_time')

fig, ax = plt.subplots(figsize=(16, 8))

for idx, (_, row) in enumerate(lemi_data.iterrows()):
    ax.barh(idx, 
            (row['end_time'] - row['start_time']).total_seconds() / 86400,
            left=row['start_time'], 
            height=0.8, 
            color='red', 
            alpha=0.6,
            edgecolor='darkred',
            linewidth=0.5)
    
    # Add site name label
    ax.text(row['start_time'], idx, f"  {row['station_id']}", 
            va='center', ha='right', fontsize=8, color='black')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Site', fontsize=12, fontweight='bold')
ax.set_title('LEMI-424 Recording Timeline (for Remote Reference Selection)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')
ax.set_yticks(range(len(lemi_data)))
ax.set_yticklabels(lemi_data['station_id'].values, fontsize=8)

# Format x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
fig.savefig(DataPaths.FIGURES_DIR / 'lemi_recording_timeline.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 3. LEMI-424 recording timeline showing {len(lemi_data)} sites sorted by start date. "
        f"These sites recorded 2016-2017, separate from EDL deployments. "
        f"Note VIC065R and VIC067R have significantly longer durations (~210 and ~161 days) "
        f"suitable for use as remote reference stations.")

## 4. Statistical Analysis (First Hour)

Load first hour from each site and calculate statistics for QA/QC.

In [ ]:
# Function to get statistics from first hour
def get_site_statistics(mth5_file, duration_hours=1):
    """Calculate statistics from first N hours of recording."""
    try:
        df = read_station(mth5_file, channels='magnetic', verbose=False)
        
        # Get first N hours
        sample_rate = len(df) / ((df.index[-1] - df.index[0]).total_seconds() / 3600)
        n_samples = int(duration_hours * 3600 * sample_rate)
        df_sample = df.iloc[:n_samples]
        
        # Calculate statistics
        stats = {
            'bx_mean': df_sample['BX'].mean(),
            'by_mean': df_sample['BY'].mean(),
            'bz_mean': df_sample['BZ'].mean(),
            'bx_std': df_sample['BX'].std(),
            'by_std': df_sample['BY'].std(),
            'bz_std': df_sample['BZ'].std(),
            'n_samples': len(df_sample)
        }
        
        # Derived quantities
        stats['h_field'] = np.sqrt(stats['bx_mean']**2 + stats['by_mean']**2)
        stats['total_field'] = np.sqrt(stats['bx_mean']**2 + stats['by_mean']**2 + stats['bz_mean']**2)
        
        return stats
        
    except Exception as e:
        print(f"Error loading {Path(mth5_file).stem}: {e}")
        return None

print("Loading first hour from all sites for statistical analysis...")
print("This may take a few minutes...\n")

In [ ]:
# Calculate statistics for all sites
stats_list = []

for idx, row in df_meta.iterrows():
    station_id = row['station_id']
    mth5_file = Path(row['mth5_file'])
    
    print(f"[{idx+1}/{len(df_meta)}] {station_id}...", end=' ')
    
    stats = get_site_statistics(mth5_file)
    
    if stats:
        stats['station_id'] = station_id
        stats['instrument'] = row['instrument']
        stats_list.append(stats)
        print("✓")
    else:
        print("✗")

df_stats = pd.DataFrame(stats_list)

print(f"\n✓ Successfully calculated statistics for {len(df_stats)}/{len(df_meta)} sites")

In [ ]:
# Merge statistics with metadata
df_full = df_meta.merge(df_stats, on=['station_id', 'instrument'], how='left')

# Save combined data
output_file = DataPaths.PROCESSED_DATA_DIR / 'qaqc' / 'site_statistics.csv'
output_file.parent.mkdir(parents=True, exist_ok=True)
df_full.to_csv(output_file, index=False)

print(f"Saved statistics to: {output_file}")
print(f"\nPreview:")
df_full[['station_id', 'instrument', 'bx_mean', 'by_mean', 'bz_mean', 
         'h_field', 'total_field']].head(10)

## 5. Statistical Plots

In [ ]:
# Plot mean magnetic field values
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

channels = ['bx', 'by', 'bz']
colors = {'EDL': 'blue', 'LEMI-424': 'red'}

sites_with_data = df_full.dropna(subset=['bx_mean']).copy()
sites_with_data = sites_with_data.sort_values('station_id')

for i, ch in enumerate(channels):
    ax = axes[i]
    
    # Plot by instrument
    for instrument in ['EDL', 'LEMI-424']:
        subset = sites_with_data[sites_with_data['instrument'] == instrument]
        
        if len(subset) > 0:
            mask = sites_with_data['instrument'] == instrument
            x_instrument = [i for i, m in enumerate(mask) if m]
            y = subset[f'{ch}_mean'].values
            ax.scatter(x_instrument, y, c=colors[instrument], alpha=0.7, s=60, 
                      label=f'{instrument} (n={len(subset)})', zorder=3)
    
    # Add median line
    median_val = sites_with_data[f'{ch}_mean'].median()
    ax.axhline(median_val, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
              label=f'Median: {median_val:.1f} nT', zorder=2)
    
    ax.set_ylabel(f'{ch.upper()} Mean (nT)', fontsize=12, fontweight='bold')
    ax.set_title(f'{ch.upper()} Component', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3, zorder=1)
    ax.set_xlim(-0.5, len(sites_with_data) - 0.5)
    
    if i == len(channels) - 1:  # Bottom plot
        ax.set_xlabel('Site', fontsize=12, fontweight='bold')
        tick_positions = range(len(sites_with_data))
        tick_labels = sites_with_data['station_id'].values
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=8, ha='right')
    else:
        ax.set_xticklabels([])

plt.suptitle('Mean Magnetic Field Values - All Sites (First Hour)', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
fig.savefig(DataPaths.FIGURES_DIR / 'qaqc' / 'mean_field_values.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 4. Mean magnetic field values for all {len(sites_with_data)} sites (first hour of recording). "
        f"Blue: EDL. Red: LEMI-424. Black dashed line shows overall median.")

In [ ]:
# Plot standard deviations
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for i, ch in enumerate(channels):
    ax = axes[i]
    
    for instrument in ['EDL', 'LEMI-424']:
        subset = sites_with_data[sites_with_data['instrument'] == instrument]
        
        if len(subset) > 0:
            mask = sites_with_data['instrument'] == instrument
            x_instrument = [i for i, m in enumerate(mask) if m]
            y = subset[f'{ch}_std'].values
            ax.scatter(x_instrument, y, c=colors[instrument], alpha=0.7, s=60,
                      label=f'{instrument} (n={len(subset)})', zorder=3)
    
    ax.set_ylabel(f'{ch.upper()} Std Dev (nT)', fontsize=12, fontweight='bold')
    ax.set_title(f'{ch.upper()} Component', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3, zorder=1)
    ax.set_xlim(-0.5, len(sites_with_data) - 0.5)
    
    if i == len(channels) - 1:
        ax.set_xlabel('Site', fontsize=12, fontweight='bold')
        tick_positions = range(len(sites_with_data))
        tick_labels = sites_with_data['station_id'].values
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=8, ha='right')
    else:
        ax.set_xticklabels([])

plt.suptitle('Standard Deviation - All Sites (First Hour)', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
fig.savefig(DataPaths.FIGURES_DIR / 'qaqc' / 'std_field_values.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 5. Standard deviation of magnetic field components. "
        f"Shows natural field variations and instrument noise levels.")

In [ ]:
# Plot horizontal and total field
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Horizontal field
ax = axes[0]
for instrument in ['EDL', 'LEMI-424']:
    subset = sites_with_data[sites_with_data['instrument'] == instrument]
    
    if len(subset) > 0:
        mask = sites_with_data['instrument'] == instrument
        x_instrument = [i for i, m in enumerate(mask) if m]
        y = subset['h_field'].values
        ax.scatter(x_instrument, y, c=colors[instrument], alpha=0.7, s=60,
                  label=f'{instrument} (n={len(subset)})', zorder=3)

ax.set_ylabel('Horizontal Field (nT)', fontsize=12, fontweight='bold')
ax.set_title('Horizontal Field (H = √(Bx² + By²))', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3, zorder=1)
ax.set_xticklabels([])
ax.set_xlim(-0.5, len(sites_with_data) - 0.5)

# Total field
ax = axes[1]
for instrument in ['EDL', 'LEMI-424']:
    subset = sites_with_data[sites_with_data['instrument'] == instrument]
    
    if len(subset) > 0:
        mask = sites_with_data['instrument'] == instrument
        x_instrument = [i for i, m in enumerate(mask) if m]
        y = subset['total_field'].values
        ax.scatter(x_instrument, y, c=colors[instrument], alpha=0.7, s=60,
                  label=f'{instrument} (n={len(subset)})', zorder=3)

median_total = sites_with_data['total_field'].median()
ax.axhline(median_total, color='black', linestyle='--', linewidth=1.5, alpha=0.7,
          label=f'Median: {median_total:.1f} nT', zorder=2)

ax.set_ylabel('Total Field (nT)', fontsize=12, fontweight='bold')
ax.set_xlabel('Site', fontsize=12, fontweight='bold')
ax.set_title('Total Field (F = √(Bx² + By² + Bz²))', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3, zorder=1)
ax.set_xlim(-0.5, len(sites_with_data) - 0.5)

tick_positions = range(len(sites_with_data))
tick_labels = sites_with_data['station_id'].values
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=90, fontsize=8, ha='right')

plt.suptitle('Derived Field Quantities - All Sites', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
fig.savefig(DataPaths.FIGURES_DIR / 'qaqc' / 'derived_field_quantities.png', dpi=300, bbox_inches='tight')
plt.show()

caption(f"Figure 6. Horizontal and total field strength for all sites.")

## 6. AGRF25 Model Comparison

Compare measured fields to Australian Geomagnetic Reference Field model (future notebook).

In [ ]:
print("AGRF25 comparison will be implemented in notebook 03_agrf_comparison.ipynb")
print("\nThis will include:")
print("  - AGRF25 calculation at each site")
print("  - Residual analysis (measured - model)")
print("  - Identification of secular variation")
print("  - Detection of crustal magnetic anomalies")

## 7. QA/QC Summary

In [ ]:
# Identify potential issues
issues = []

# Check for short recordings
short_threshold = 15  # days
short_recordings = df_full[df_full['duration_days'] < short_threshold]
if len(short_recordings) > 0:
    issues.append(f"⚠️  {len(short_recordings)} sites with duration < {short_threshold} days")
    for _, row in short_recordings.iterrows():
        issues.append(f"    - {row['station_id']}: {row['duration_days']:.1f} days")

# Check for missing statistics (failed loads)
missing_stats = df_full[df_full['bx_mean'].isna()]
if len(missing_stats) > 0:
    issues.append(f"⚠️  {len(missing_stats)} sites failed to load for statistics")
    for _, row in missing_stats.iterrows():
        issues.append(f"    - {row['station_id']}")

# Check for high noise (large std)
noise_threshold = 50  # nT
high_noise = df_full[(df_full['bx_std'] > noise_threshold) | 
                     (df_full['by_std'] > noise_threshold) | 
                     (df_full['bz_std'] > noise_threshold)]
if len(high_noise) > 0:
    issues.append(f"⚠️  {len(high_noise)} sites with high noise (std > {noise_threshold} nT)")
    for _, row in high_noise.iterrows():
        issues.append(f"    - {row['station_id']}: BX={row['bx_std']:.1f}, BY={row['by_std']:.1f}, BZ={row['bz_std']:.1f} nT")

# Print summary
print("QA/QC SUMMARY")
print("=" * 80)

if len(issues) == 0:
    print("✓ All sites passed QA/QC checks")
else:
    print(f"Found {len(issues)} potential issues:\n")
    for issue in issues:
        print(issue)

print("\n" + "=" * 80)
print(f"Total sites analyzed: {len(df_full)}")
print(f"Sites with complete statistics: {df_full['bx_mean'].notna().sum()}")
print(f"\nRecommendation: {df_full['bx_mean'].notna().sum() - len(high_noise) - len(short_recordings)} sites are suitable for MT processing")

## Summary

**Completed**:
- ✓ Loaded metadata for 100 sites (81 EDL + 19 LEMI-424)
- ✓ Created site location map
- ✓ Generated recording timelines for remote reference selection
- ✓ Calculated statistical metrics for all sites
- ✓ Generated QA/QC plots and summary

**Next Steps**:
1. AGRF25 model comparison (notebook 03)
2. Select representative sites for MT processing
3. Identify optimal remote reference pairs
4. Begin impedance tensor estimation